In [0]:
from pyspark.sql.functions import lit

catalog_name = "multi_flow_demo"

bronze_schema = "multi_flow_1_bronze"
silver_schema = "multi_flow_2_silver"
gold_schema = "multi_flow_3_gold"

bsh_volume = "bright_home_orders"
lms_volume = "lumina_sports_orders"
nso_volume = "northstar_outfitters_orders"

source_path = "/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders"
sink_path = f"/Volumes/{catalog_name}/{bronze_schema}"

date_on_file = "2025-11-01"
bsh_file_name = f"bsh_orders_{date_on_file}.csv"
lms_file_name = f"lms_orders_{date_on_file}.csv"
nso_file_name = f"nso_orders_{date_on_file}.json"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{bronze_schema}.{bsh_volume}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{bronze_schema}.{lms_volume}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{bronze_schema}.{nso_volume}")

In [0]:
from pyspark.sql.functions import lit

dbutils.fs.cp(f"{source_path}/{bsh_volume}/{bsh_file_name}", f"{sink_path}/{bsh_volume}/{bsh_file_name}")
dbutils.fs.cp(f"{source_path}/{lms_volume}/{lms_file_name}", f"{sink_path}/{lms_volume}/{lms_file_name}")
dbutils.fs.cp(f"{source_path}/{nso_volume}/{nso_file_name}", f"{sink_path}/{nso_volume}/{nso_file_name}")

In [0]:
## Check file
sql_result = (
    spark.sql(f"LIST '{sink_path}/{bsh_volume}/'")
    .withColumn('volume', lit(f"{bsh_volume}"))
)
sql_result = (
    sql_result.unionAll(
        spark.sql(f"LIST '{sink_path}/{lms_volume}/'")
        .withColumn('volume', lit(f"{lms_volume}"))
    )
)
sql_result = (
    sql_result.unionAll(
        spark.sql(f"LIST '{sink_path}/{nso_volume}/'")
        .withColumn('volume', lit(f"{nso_volume}")))
).orderBy("volume")

display(sql_result)

In [0]:
sql_query_result = (
    spark.sql(f"""
        SELECT '{bsh_volume}' as volume_name,
            COUNT(*) as total_rows,
            _metadata.file_name as file_name
        FROM read_files('{sink_path}/{bsh_volume}/{bsh_file_name}')
        GROUP BY _metadata.file_name
        UNION ALL
        SELECT '{lms_volume}' as volume_name,
            COUNT(*) as total_rows,
            _metadata.file_name as file_name
        FROM read_files('{sink_path}/{lms_volume}/{lms_file_name}')
        GROUP BY _metadata.file_name
        UNION ALL
        SELECT '{nso_volume}' as volume_name,
            COUNT(*) as total_rows,
            _metadata.file_name as file_name
        FROM read_files('{sink_path}/{nso_volume}/{nso_file_name}')
        GROUP BY _metadata.file_name 
    """)
)

display(sql_query_result)

In [0]:
spark.sql(f"""
          DESCRIBE SELECT * 
          FROM read_files('{sink_path}/{bsh_volume}/{bsh_file_name}')
""").display()

In [0]:
spark.sql("""
        SELECT source_file,
                COUNT(*) total_rows
        FROM multi_flow_demo.multi_flow_1_bronze.orders_bronze_flow_demo
        GROUP BY source_file
""").display()